In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# LOADING THE LABELED DATA
try:
    df = pd.read_excel(r'C:\Users\Harsh Datt\Data Science\2nd Classification\labeled_final_dataset.xlsx')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'labeled_final_dataset.xlsx' not found.")
    exit()

# SEPARATING FEATURES (X) AND TARGET (y)
y = df['BPL_Target']

# Identifying direct income/threshold columns to drop (preventing Target Leakage)
# We only want to drop the raw 'memberVerifiedRange' columns.
# We look for columns that start with 'in_' AND contain numbers (e.g., 'in_180001-300000' or 'in_0').
income_cols = [col for col in df.columns if col.startswith('in_') and any(char.isdigit() for char in col) or col.startswith('incometaxthreshold_')]

# Metadata columns to drop
metadata_cols = ['hasfamilyid', 'BPL_Target', 'district', 'blocktown', 'wardvillage', 'r_u', 'familyRange']

# Noisy proxy columns to drop (updated to match the new 'is_<category>' naming convention)
noisy_cols = ['is_Child', 'is_Housewife', 'is_Senior Citizen', 'is_Student', 'is_Farmer', 'is_Labour', 'is_Pensioner/Retired']

cols_to_drop = metadata_cols + noisy_cols + income_cols 
# Comment '+ income_cols' and the lines of code above the comment in the 100-District_Wise.ipynb to add the Per Member Salary to the training dataset

# Isolating the true proxy features
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

print(f"Total features selected for training: {X.shape[1]}")

# STRATIFYING TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"Training set size: {X_train.shape[0]} families")
print(f"Testing set size: {X_test.shape[0]} families")

# BUILDING THE BALANCED PIPELINES
# StandardScaler is used for distance-based algorithms (LR, SVM).
# Tree/Probability based algorithms (NB, RF) use unscaled data.
pipelines = {
    "Logistic Regression (Penalized)": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
    ]),
    "SVM (Linear Kernel)": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='linear', class_weight='balanced', random_state=42))
    ]),
    "Bernoulli Naive Bayes": Pipeline([
        ('classifier', BernoulliNB())
    ]),
    "Random Forest (Shallow Trees)": Pipeline([
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=3, class_weight='balanced', random_state=42))
    ])
}

# STRATIFIED CROSS-VALIDATION SETTINGS
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
leaderboard_results = []

# LOOPING THROUGH EACH ALGORITHM FOR EVALUATION
for name, pipeline in pipelines.items():
    print(f"  MODEL: {name.upper()}")
    
    # STRATIFIED CROSS-VALIDATION
    print("\n--- Running 5-Fold Stratified Cross-Validation ---")
    cv_results = cross_validate(pipeline, X_train, y_train, cv=cv_strategy, scoring=['f1_macro', 'accuracy'])
    
    mean_f1 = np.mean(cv_results['test_f1_macro'])
    std_f1 = np.std(cv_results['test_f1_macro'])
    print(f"Cross-Validation Macro F1-Score: {mean_f1:.4f} (± {std_f1:.4f})")
    if std_f1 > 0.1:
        print("Note: High standard deviation indicates the model's performance fluctuates depending on the data slice.")

    # FINAL TRAINING & EVALUATION ON UNSEEN TEST DATA
    print("\n--- Final Model Evaluation on Unseen Test Data ---")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # DYNAMICALLY SAVING EACH MODEL
    # CleanING the name to make it a valid, lowercase file name without spaces or parentheses
    safe_file_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    model_filename = f"2nd_{safe_file_name}_balanced.joblib"
    
    # Saving the pipeline to your hard drive
    joblib.dump(pipeline, model_filename)
    print(f"💾 Successfully saved model as: '{model_filename}'")
    # ==========================================
    
    # Calculating metrics for the leaderboard
    holdout_macro_f1 = f1_score(y_test, y_pred, average='macro')
    holdout_acc = accuracy_score(y_test, y_pred)
    
    leaderboard_results.append({
        "Algorithm": name,
        "CV Train F1": round(mean_f1, 4),
        "CV Variance": round(std_f1, 4),
        "Holdout F1": round(holdout_macro_f1, 4),
        "Holdout Acc": round(holdout_acc, 4)
    })

    # Displaying the classification report and confusion matrix
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Non-BPL (0)', 'BPL (1)']))

    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred)
    print(f"True Negatives (Correct Non-BPL): {cm[0][0]}")
    print(f"False Positives (Fraudulent BPL Guesses): {cm[0][1]}")
    print(f"False Negatives (Missed BPL Families): {cm[1][0]}")
    print(f"True Positives (Correct BPL): {cm[1][1]}")
    print("\n")

# PRINTING THE CONSOLIDATED LEADERBOARD
print("                 🏆 OVERALL ALGORITHM LEADERBOARD 🏆                 ")
# Sorting by the F1-Score achieved on the 20% Unseen Holdout data
leaderboard_df = pd.DataFrame(leaderboard_results).sort_values(by="Holdout F1", ascending=False)
print(leaderboard_df.to_string(index=False))

Dataset loaded successfully.
Total features selected for training: 29
Training set size: 1840 families
Testing set size: 461 families
  MODEL: LOGISTIC REGRESSION (PENALIZED)

--- Running 5-Fold Stratified Cross-Validation ---
Cross-Validation Macro F1-Score: 0.8633 (± 0.0122)

--- Final Model Evaluation on Unseen Test Data ---
💾 Successfully saved model as: '2nd_logistic_regression_penalized_balanced.joblib'

Classification Report:
              precision    recall  f1-score   support

 Non-BPL (0)       0.96      0.75      0.84       240
     BPL (1)       0.78      0.97      0.86       221

    accuracy                           0.85       461
   macro avg       0.87      0.86      0.85       461
weighted avg       0.88      0.85      0.85       461

Confusion Matrix:
True Negatives (Correct Non-BPL): 180
False Positives (Fraudulent BPL Guesses): 60
False Negatives (Missed BPL Families): 7
True Positives (Correct BPL): 214


  MODEL: SVM (LINEAR KERNEL)

--- Running 5-Fold Stratifie